In [715]:
# 그래프관련 import 
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus']=False

# 프로그램 
import pandas as pd
import numpy as np
from sklearn import svm,metrics
from sklearn.neighbors import KNeighborsClassifier # 분류 클래스
from sklearn.neighbors import KNeighborsRegressor # 예측 클래스
from sklearn.linear_model import LinearRegression # 예측 클래스
from sklearn.preprocessing import PolynomialFeatures # 다항특성 자동생성 라이브러리
from sklearn.metrics import mean_absolute_error # 평균절대오차 오차범위 클래스
from sklearn.preprocessing import StandardScaler  # 정규화 클래스
from sklearn.model_selection import train_test_split

In [716]:
df = pd.read_csv('data/perch_full.csv')
df.head(2)

,length,height,width
0,8.4,2.11,1.41
1,13.7,3.53,2.00


In [717]:
df.shape

(56, 3)

In [718]:
# 무게 데이터
perch_weight = [5.9, 32.0, 40.0, 51.5, 70.0, 100.0, 78.0, 80.0, 85.0, 85.0,
110.0, 115.0, 125.0, 130.0, 120.0, 120.0, 130.0, 135.0, 110.0,
130.0, 150.0, 145.0, 150.0, 170.0, 225.0, 145.0, 188.0, 180.0,
197.0, 218.0, 300.0, 260.0, 265.0, 250.0, 250.0, 300.0, 320.0,
514.0, 556.0, 840.0, 685.0, 700.0, 700.0, 690.0, 900.0, 650.0,
820.0, 850.0, 900.0, 1015.0, 820.0, 1100.0, 1000.0, 1100.0,
1000.0, 1000.0]

In [719]:
len(perch_weight)

56

In [720]:
# 학습데이터, 테스트데이터 분리
train_data, test_data, train_label, test_label = train_test_split(df,perch_weight, random_state=42)
train_data.shape

(42, 3)

In [ ]:
# 다항특성 값을 추가 
# length, height, width 
# length**2, height**2, width**2 로 변경했으나 자동생성 라이브러리 사용

# degree = 5 (5배만큼 특성값 늘림) include_bias=False (1은 재외하고 실행)
# degree 를 무작정 늘린다고 좋은 거 아님. 마이너스 정확도도 나옴 
# degree를 늘리면 과대적합에 가까움. 보통은 2,3배 정도 늘림. 
# (42.... : 이 부분보다 조금 낮추면 정확도가 높아짐. 특성값만 늘리면 정확도가 과대적합에 가깝게 실행됨

poly = PolynomialFeatures(degree=4,include_bias=False) 
# poly = PolynomialFeatures(degree=20,include_bias=False) # degree 를 무작정 늘린다고 좋은 거 아님. 마이너스 정확도도 나옴

In [722]:
# 다항특성추가 - PolynomialFeatures 
poly.fit([[2,3]])
train_poly = poly.transform([[2,3]])
train_poly

array([[ 2.,  3.,  4.,  6.,  9.,  8., 12., 18., 27., 16., 24., 36., 54.,
        81.]])

In [723]:
poly.fit(train_data)
train_poly = poly.transform(train_data)
test_poly = poly.transform(test_data)
print(train_poly.shape)

(42, 34)


In [724]:
train_data[:5]

,length,height,width
8,19.6,5.14,3.04
17,22.0,5.88,3.52
6,18.7,5.20,3.12
4,17.4,4.59,2.94
41,36.0,10.61,6.74


In [725]:
train_poly[:2]

array([[1.96000000e+01, 5.14000000e+00, 3.04000000e+00, 3.84160000e+02,
        1.00744000e+02, 5.95840000e+01, 2.64196000e+01, 1.56256000e+01,
        9.24160000e+00, 7.52953600e+03, 1.97458240e+03, 1.16784640e+03,
        5.17824160e+02, 3.06261760e+02, 1.81135360e+02, 1.35796744e+02,
        8.03155840e+01, 4.75018240e+01, 2.80944640e+01, 1.47578906e+05,
        3.87018150e+04, 2.28897894e+04, 1.01493535e+04, 6.00273050e+03,
        3.55025306e+03, 2.66161618e+03, 1.57418545e+03, 9.31035750e+02,
        5.50651494e+02, 6.97995264e+02, 4.12822102e+02, 2.44159375e+02,
        1.44405545e+02, 8.54071706e+01],
       [2.20000000e+01, 5.88000000e+00, 3.52000000e+00, 4.84000000e+02,
        1.29360000e+02, 7.74400000e+01, 3.45744000e+01, 2.06976000e+01,
        1.23904000e+01, 1.06480000e+04, 2.84592000e+03, 1.70368000e+03,
        7.60636800e+02, 4.55347200e+02, 2.72588800e+02, 2.03297472e+02,
        1.21701888e+02, 7.28555520e+01, 4.36142080e+01, 2.34256000e+05,
        6.26102400e+04,

In [726]:
print(poly.get_feature_names_out()) # 어떠한 방식으로 만들어졌는지 확인 (다항특성추가 만든 형태 확인) 

['length' ' height' ' width' 'length^2' 'length  height' 'length  width'
 ' height^2' ' height  width' ' width^2' 'length^3' 'length^2  height'
 'length^2  width' 'length  height^2' 'length  height  width'
 'length  width^2' ' height^3' ' height^2  width' ' height  width^2'
 ' width^3' 'length^4' 'length^3  height' 'length^3  width'
 'length^2  height^2' 'length^2  height  width' 'length^2  width^2'
 'length  height^3' 'length  height^2  width' 'length  height  width^2'
 'length  width^3' ' height^4' ' height^3  width' ' height^2  width^2'
 ' height  width^3' ' width^4']


In [727]:
# 3개의 특성으로 예측 
# lr = LinearRegression()
# lr.fit(train_data,train_label)

# predict = lr.predict(test_data)
# print("예측 :", predict)

# score = lr.score(train_data,train_label)
# score2 = lr.score(test_data,test_label)
# print('train 정확도 :', score)
# print('test 정확도 :', score2) # train 정확도가 낮으면 조금 이상한 현상이고, 마이너스 값이 있다면 예측이 잘못된 것임

In [728]:
# degree = 2 로 선정하여 특성을 9개로 변경해서 예측
lr = LinearRegression()
lr.fit(train_poly,train_label)

predict = lr.predict(test_poly)
print("예측 :", predict)

score = lr.score(train_poly,train_label)
score2 = lr.score(test_poly,test_label)
print('train 정확도 :', score)
print('test 정확도 :', score2) # test 데이터 예측 정확도가 낮기에 원본 3개로 돌라는 것보다는 정확하다는 의미, 할 때마다 예측 값이 다른 이유는 랜덤으로 진행하기에 할 때마다 다른 예측값이 나옴
# 과대적합, 과소적합


예측 : [  27.6526997    61.33148601  255.5563934   126.29512923  135.28727748
  780.45784191  336.71746318  180.46622142  875.77899089  101.55406992
 1168.8311871    51.08787117  256.16201835  273.36562632]
train 정확도 : 0.9950570894726734
test 정확도 : 0.9755705435112506
